# 01 — Offre de bus par arrêt

**Rôle dans le pipeline :** pour un jour de référence, compter combien de bus passent à chaque arrêt, pour chaque réseau GTFS listé dans `parametres.py`.

| Lit | Produit |
|---|---|
| `data/bronze/gtfs/<réseau>/*.txt` | `data/silver/offre_arrets.parquet` |

Les fichiers Bronze ne sont jamais modifiés. Le résultat est écrit dans Silver pour que les notebooks suivants n'aient pas à relire les GTFS (le seul `stop_times.txt` de Kar'Ouest fait 7,6 Mo).

In [1]:
import pandas as pd
from parametres import GTFS_DIR, SILVER, RESEAUX, DATE_REF

SILVER.mkdir(parents=True, exist_ok=True)

### Fonction `offre_par_arret` — un réseau, une date

Même calcul que les cellules 30–31 du notebook Fabric, déjà validé par un test de non-régression : il retrouvait à l'identique le calcul manuel (Kar'Ouest, mardi 20/10/2026 : **3 services, 1 920 courses, 2 264 arrêts desservis**).

Services actifs ce jour-là = ceux de `calendar.txt` (bon jour de semaine, dans la période de validité), **plus** les ajouts (`exception_type = 1`), **moins** les suppressions (`exception_type = 2`) de `calendar_dates.txt`.

Deux ajouts pour la réplicabilité :
- certains réseaux n'ont **que** `calendar_dates.txt` : la fonction accepte qu'un des deux fichiers calendrier manque ;
- les identifiants sont lus **comme du texte** : selon les réseaux, `stop_id` vaut `1234` ou `"KO_1234"`, et mélanger nombres et textes casserait les jointures et l'écriture du fichier Silver.

In [2]:
IDS_TEXTE = {"stop_id": str, "trip_id": str, "route_id": str, "service_id": str}


def lire_si_present(dossier, nom):
    chemin = dossier / nom
    return pd.read_csv(chemin, dtype=IDS_TEXTE) if chemin.exists() else None


def offre_par_arret(dossier, date, reseau):
    stops = pd.read_csv(dossier / "stops.txt", dtype=IDS_TEXTE)
    stop_times = pd.read_csv(dossier / "stop_times.txt", dtype=IDS_TEXTE, usecols=["trip_id", "stop_id"])
    trips = pd.read_csv(dossier / "trips.txt", dtype=IDS_TEXTE)
    calendar = lire_si_present(dossier, "calendar.txt")
    cal_dates = lire_si_present(dossier, "calendar_dates.txt")

    # Services actifs ce jour-là (calendrier + exceptions)
    services = set()
    if calendar is not None:
        jour = pd.Timestamp(str(date)).day_name().lower()  # ex. "tuesday"
        actifs = calendar[
            (calendar["start_date"] <= date)
            & (calendar["end_date"] >= date)
            & (calendar[jour] == 1)
        ]
        services = set(actifs["service_id"])
    if cal_dates is not None:
        exc = cal_dates[cal_dates["date"] == date]
        services |= set(exc[exc["exception_type"] == 1]["service_id"])
        services -= set(exc[exc["exception_type"] == 2]["service_id"])

    # Passages et lignes par arrêt
    trips_jour = trips[trips["service_id"].isin(services)]
    passages = stop_times.merge(trips_jour[["trip_id", "route_id"]], on="trip_id")
    resultat = passages.groupby("stop_id").agg(
        nb_passages=("trip_id", "count"),
        nb_lignes=("route_id", "nunique"),
    ).reset_index()
    resultat = resultat.merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
    resultat["reseau"] = reseau

    print(f"{reseau} : {len(services)} services, {len(trips_jour)} courses, {len(resultat)} arrêts desservis")
    return resultat

### Tous les réseaux de `parametres.py`

Un réseau dont le dossier est vide est **signalé et ignoré** : le pipeline produit quand même un résultat avec les données disponibles (principe CORE du brief).

In [3]:
offres = []
for reseau, sous_dossier in RESEAUX.items():
    dossier = GTFS_DIR / sous_dossier
    if not (dossier / "stops.txt").exists():
        print(f"⚠️ {reseau} : pas de GTFS dans {dossier} — réseau ignoré")
        continue
    offres.append(offre_par_arret(dossier, DATE_REF, reseau))

offre_arrets = pd.concat(offres, ignore_index=True)

Kar'Ouest : 3 services, 1920 courses, 2264 arrêts desservis
Car Jaune : 1 services, 566 courses, 319 arrêts desservis


### Contrôles

- **Par réseau :** nombre d'arrêts desservis, passages cumulés, médiane. Pour Kar'Ouest, la médiane attendue est **18 passages** par arrêt (constat de lundi).
- **Coordonnées manquantes :** un arrêt sans latitude/longitude ne pourrait pas être placé sur la carte ni croisé avec la population. On attend **0**.
- **Top 10 :** les arrêts les plus desservis doivent être des pôles connus (Plateau Caillou pour Kar'Ouest).

In [4]:
print(offre_arrets.groupby("reseau").agg(
    arrets=("stop_id", "count"),
    passages=("nb_passages", "sum"),
    mediane_passages=("nb_passages", "median"),
))
print("\nArrêts sans coordonnées :", offre_arrets[["stop_lat", "stop_lon"]].isna().any(axis=1).sum())
offre_arrets.sort_values("nb_passages", ascending=False).head(10)

           arrets  passages  mediane_passages
reseau                                       
Car Jaune     319      8849              24.0
Kar'Ouest    2264     57155              18.0

Arrêts sans coordonnées : 0


,stop_id,nb_passages,nb_lignes,stop_name,stop_lat,stop_lon,reseau
1321,521_r,222,12,Les Arcades,-21.023361,55.266052,Kar'Ouest
1482,5_r,222,12,AFPAR Plateau Caillou,-21.021336,55.265734,Kar'Ouest
1852,771_r,222,12,Rond Point Plateau Caillou,-21.031764,55.269446,Kar'Ouest
1320,521_a,221,12,Les Arcades,-21.022682,55.266125,Kar'Ouest
1481,5_a,221,12,AFPAR Plateau Caillou,-21.020693,55.265702,Kar'Ouest
313,1222_a,221,12,Eglise Plateau Caillou,-21.026565,55.265430,Kar'Ouest
314,1222_r,210,11,Eglise Plateau Caillou,-21.027060,55.265292,Kar'Ouest
1039,386_r,210,11,Fleurimont,-21.029621,55.265298,Kar'Ouest
1432,577_r,210,11,Lycée de Plateau Caillou,-21.029178,55.270103,Kar'Ouest
1038,386_a,209,11,Fleurimont,-21.030105,55.266292,Kar'Ouest


In [5]:
offre_arrets.to_parquet(SILVER / "offre_arrets.parquet", index=False)
print("Écrit :", SILVER / "offre_arrets.parquet", "|", len(offre_arrets), "arrêts")

Écrit : C:\Users\nicol\Documents\Epitech\hackathon-citiz\data\silver\offre_arrets.parquet | 2583 arrêts
